In [ ]:
import requests
import pandas as pd
import io
import os
from datetime import datetime

# --- 1. Date Logic ---
now = datetime.now()
go_back = 3  # Adjust this to 1 if you want the immediate previous month
last_month_date = datetime(now.year, now.month, 1) - pd.DateOffset(months=go_back)

L_YEAR = last_month_date.year
L_MONTH_NAME = last_month_date.strftime("%B")
L_MONTH_STR = last_month_date.strftime("%m")
DATE_KEY = f"{L_YEAR}-{L_MONTH_STR}"

# --- 2. Configuration ---
MASTER_FILE = "Export-Import Customs_31072025.xlsx"
API_TOKEN = "fHs4RbfpSE57qfYM"
BASE_URL = "https://stats.customs.gov.kh/api/file/download"

# Mapping target categories to their specific files and sheet names
FILE_MAP = {
    "Export": {
        "files": [f"{DATE_KEY}-TSC-EN-EX.xlsx", f"{DATE_KEY}-TSCC-EN-EX.xlsx"], 
        "sheet": "Data_Matrix_Export "
    },
    "Import": {
        "files": [f"{DATE_KEY}-TSC-EN-IM.xlsx", f"{DATE_KEY}-TSCC-EN-IM.xlsx"], 
        "sheet": "DataMatrix_Import "
    }
}

print(f"📅 Targeting Data: {L_MONTH_NAME} {L_YEAR} (Key: {DATE_KEY})")

def clean_raw_data(content):
    """Extracts columns 1 and 4 from the source Excel and removes headers/footers."""
    df = pd.read_excel(io.BytesIO(content), header=None)
    # iloc[6:-3] skips the top 6 rows and bottom 3 rows of the source file
    df_clean = df.iloc[6:-2].reset_index(drop=True)
    return df_clean[[1, 4]].dropna()

def transform_to_standard(df, title, filename):
    """Converts raw scrapped data into the Master File's standardized format."""
    standard_rows = []
    is_tsc = "TSC-" in filename
    indicator_type = "Main market" if is_tsc else "Main product"

    for _, row in df.iterrows():
        desc = str(row.iloc[0]).strip()
        val = row.iloc[1]

        # Filter out sub-totals or headers found within the data body
        if desc.lower() in ["country", "description", "total", "grand total"] or desc.upper().startswith("TOTAL"):
            continue

        standard_rows.append({
            "No.": 0, "Tittle ": title, "Country": "Cambodia", "Update frequency ": "Monthly",
            "Status": "Real", "Yearly": L_YEAR, "Monthly": L_MONTH_NAME,
            "Indecator": indicator_type, "Sub1": desc if is_tsc else ".", "Sub2": desc if not is_tsc else ".",
            "Sub3": ".", "Sub4": ".", "Sub5": ".", "Sub6": ".", "Unit": "(Value in Thousand US $)",
            "Value": val, "Accesss Date": datetime.now().strftime("%m/%d/%Y"), "Pubilsh Date": "",
            "Link(if avilable)": "https://stats.customs.gov.kh/en/publication", "Note": "Provisional"
        })
    return pd.DataFrame(standard_rows)

def main():
    headers = {"User-Agent": "Mozilla/5.0", "x-api-token": API_TOKEN}

    if not os.path.exists(MASTER_FILE):
        print(f"❌ Error: {MASTER_FILE} not found.")
        return

    # 1. Load EVERY sheet from the workbook into a dictionary
    try:
        all_sheets = pd.read_excel(MASTER_FILE, sheet_name=None)
        print(f"📂 Loaded {len(all_sheets)} sheets from master file.")
    except Exception as e:
        print(f"❌ Error reading Excel file: {e}")
        return

    # Create a copy to hold our updates while keeping other sheets intact
    sheets_to_write = all_sheets.copy()

    for category, config in FILE_MAP.items():
        target_name = config["sheet"].strip()
        
        # Find actual sheet name (matching case and spaces)
        actual_sheet_name = next((s for s in all_sheets.keys() if s.strip() == target_name), None)

        if not actual_sheet_name:
            print(f"⚠️ Sheet '{target_name}' not found in Excel. Skipping {category}.")
            continue

        existing_df = all_sheets[actual_sheet_name]

        # --- Duplicate Check ---
        if not existing_df.empty and 'Yearly' in existing_df.columns:
            # Check if Year and Month combination already exists
            duplicate_rows = existing_df[
                (existing_df['Yearly'].astype(str).str.strip() == str(L_YEAR)) &
                (existing_df['Monthly'].astype(str).str.strip().str.lower() == L_MONTH_NAME.lower())
            ]

            if not duplicate_rows.empty:
                print(f"🛑 SKIP: Data for {L_MONTH_NAME} {L_YEAR} already exists in '{actual_sheet_name}'.")
                continue

            # Calculate next ID number
            last_no = pd.to_numeric(existing_df.iloc[:, 0], errors='coerce').max()
            last_no = 0 if pd.isna(last_no) else last_no
        else:
            last_no = 0

        # --- Scrapping Logic ---
        print(f"🚀 Processing {category}s...")
        new_data_list = []
        for fname in config["files"]:
            params = {"filePath": f"/ePay-SAN/statistics/trade/{L_YEAR}/excel/{fname}", "filename": fname}
            try:
                res = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
                if res.status_code == 200:
                    new_data_list.append(transform_to_standard(clean_raw_data(res.content), category, fname))
                    print(f"   ✅ Fetched {fname}")
                else:
                    print(f"   ❌ Failed to fetch {fname} (Status: {res.status_code})")
            except Exception as e:
                print(f"   ❌ Connection error for {fname}: {e}")

        if new_data_list:
            new_df = pd.concat(new_data_list, ignore_index=True)
            # Generate sequential "No." column
            new_df['No.'] = range(int(last_no) + 1, int(last_no) + 1 + len(new_df))

            # Append new data to the existing sheet data
            updated_df = pd.concat([existing_df, new_df], ignore_index=True)
            sheets_to_write[actual_sheet_name] = updated_df
            print(f"🎉 Successfully added {len(new_df)} rows to {actual_sheet_name}")

    # 3. Save everything back to the Master File
    try:
        with pd.ExcelWriter(MASTER_FILE, engine='openpyxl', mode='w') as writer:
            for sheet_name, df in sheets_to_write.items():
                df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"\n✅ SUCCESS: All data and original sheets written to {MASTER_FILE}")
    except PermissionError:
        print(f"\n❌ ERROR: Could not save file. Please close '{MASTER_FILE}' and try again.")
    except Exception as e:
        print(f"\n❌ ERROR saving file: {e}")

if __name__ == "__main__":
    main()

📅 Targeting Data: December 2025 (Key: 2025-12)
📂 Loaded 13 sheets from master file.
🚀 Processing Exports...
   ✅ Fetched 2025-12-TSC-EN-EX.xlsx
   ✅ Fetched 2025-12-TSCC-EN-EX.xlsx
🎉 Successfully added 42 rows to Data_Matrix_Export 
🚀 Processing Imports...
   ✅ Fetched 2025-12-TSC-EN-IM.xlsx
   ✅ Fetched 2025-12-TSCC-EN-IM.xlsx
🎉 Successfully added 42 rows to DataMatrix_Import 

✅ SUCCESS: All data and original sheets written to Export-Import Customs_31072025.xlsx


In [ ]:
import os
import io
import requests
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# ==========================================================
# CONFIG
# ==========================================================
EXCEL_PATH = r"Export-Import Customs_31072025.xlsx"
API_TOKEN  = "fHs4RbfpSE57qfYM"
BASE_URL   = "https://stats.customs.gov.kh/api/file/download"

FORCE_OVERWRITE = True  
MONTHS_BACK = 2      

DEBUG_SAVE_DOWNLOADS = True
DEBUG_DIR = "debug_downloads"

SHEET_MAP = {
    "Export": "Data_Matrix_Export",
    "Import": "Data_Matrix_Import",
}

# ==========================================================
# DATE HELPERS
# ==========================================================
def first_day_of_month(dt: datetime) -> datetime:
    return dt.replace(day=1, hour=0, minute=0, second=0, microsecond=0)

def subtract_months(dt: datetime, months: int) -> datetime:
    y, m = dt.year, dt.month
    idx = y * 12 + (m - 1) - months
    ny = idx // 12
    nm = idx % 12 + 1
    return datetime(ny, nm, 1)

# ==========================================================
# EXCEL HELPERS (preserve other sheets!)
# ==========================================================
def resolve_sheet_name(wb, desired: str) -> str:
    # Prefer exact match
    if desired in wb.sheetnames:
        return desired
    # Fallback: match ignoring trailing spaces/case
    norm = lambda s: s.strip().lower()
    for s in wb.sheetnames:
        if norm(s) == norm(desired):
            return s
    # Not found -> create new sheet with desired name
    return desired

def sheet_to_df(ws) -> pd.DataFrame:
    data = list(ws.values)
    if not data or len(data) == 0:
        return pd.DataFrame()

    headers = data[0]
    if headers is None:
        return pd.DataFrame()

    headers = [("" if h is None else str(h).strip()) for h in headers]
    rows = data[1:]
    df = pd.DataFrame(rows, columns=headers)

    # drop completely empty columns
    df = df.dropna(axis=1, how="all")
    return df

def df_to_sheet(ws, df: pd.DataFrame):
    # Clear only this sheet's content
    ws.delete_rows(1, ws.max_row)

    # Write header + rows
    for row in dataframe_to_rows(df, index=False, header=True):
        ws.append(row)

def make_updated_path(path: str) -> str:
    base, ext = os.path.splitext(path)
    return f"{base}_updated{ext}"

# ==========================================================
# DOWNLOAD
# ==========================================================
def download_excel_file(headers: dict, year: int, fname: str) -> bytes | None:
    params = {
        "filePath": f"/ePay-SAN/statistics/trade/{year}/excel/{fname}",
        "filename": fname,
    }
    try:
        res = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        size = len(res.content or b"")
        ctype = res.headers.get("Content-Type", "")

        print(f"   Status={res.status_code}  Content-Type={ctype}  Size={size}")

        if res.status_code != 200 or not res.content:
            print("   ❌ Download failed. First 200 bytes:", (res.content or b"")[:200])
            return None

        # XLSX is a ZIP container => usually starts with PK
        if not res.content.startswith(b"PK"):
            print("   ⚠️ Not an XLSX response. First 200 bytes:")
            print(res.content[:200])
            return None

        if DEBUG_SAVE_DOWNLOADS:
            os.makedirs(DEBUG_DIR, exist_ok=True)
            with open(os.path.join(DEBUG_DIR, fname), "wb") as f:
                f.write(res.content)

        return res.content
    except Exception as e:
        print(f"   ❌ Connection error: {e}")
        return None

# ==========================================================
# CLEANING (FIXED join bug)
# ==========================================================
def safe_row_text(df: pd.DataFrame, i: int) -> str:
    # Convert every cell to string safely (fix float-in-join error)
    cells = df.iloc[i].tolist()
    parts = []
    for x in cells:
        if x is None or (isinstance(x, float) and pd.isna(x)) or pd.isna(x):
            parts.append("")
        else:
            parts.append(str(x).lower())
    return " ".join(parts)

def best_numeric_col(df: pd.DataFrame) -> int | None:
    best = None
    best_count = -1
    for c in df.columns:
        nums = pd.to_numeric(df[c].astype(str).str.replace(",", "").str.strip(), errors="coerce")
        count = int(nums.notna().sum())
        if count > best_count:
            best_count = count
            best = c
    return best if best_count > 0 else None

def best_text_col(df: pd.DataFrame, exclude=None) -> int | None:
    exclude = set(exclude or [])
    best = None
    best_score = -10**9

    for c in df.columns:
        if c in exclude:
            continue
        s = df[c]
        non_empty = int(s.notna().sum())
        numeric = int(pd.to_numeric(s.astype(str).str.replace(",", "").str.strip(), errors="coerce").notna().sum())

        # prefer columns with lots of non-empty but not mostly numeric
        score = non_empty - 2 * numeric
        if score > best_score:
            best_score = score
            best = c

    return best if best_score > 0 else None

def clean_raw_data(xlsx_bytes: bytes) -> pd.DataFrame:
    try:
        df = pd.read_excel(io.BytesIO(xlsx_bytes), header=None)
        if df.empty:
            return pd.DataFrame()

        # detect table start (search first 30 rows)
        start = 0
        for i in range(min(30, len(df))):
            t = safe_row_text(df, i)
            if "country" in t or "description" in t:
                start = i + 1
                break

        # detect totals near bottom
        end = len(df)
        for i in range(len(df) - 1, max(len(df) - 30, 0), -1):
            t = safe_row_text(df, i)
            if "grand total" in t or t.strip().startswith("total"):
                end = i
                break

        df2 = df.iloc[start:end].reset_index(drop=True)
        if df2.empty:
            return pd.DataFrame()

        val_col = best_numeric_col(df2)
        desc_col = best_text_col(df2, exclude=[val_col])

        if val_col is None or desc_col is None:
            print("   ⚠️ Could not infer description/value columns.")
            return pd.DataFrame()

        out = df2[[desc_col, val_col]].copy()
        out.columns = ["desc", "value"]

        out["desc"] = out["desc"].astype(str).str.strip()
        out["value"] = pd.to_numeric(out["value"].astype(str).str.replace(",", "").str.strip(), errors="coerce")

        out = out.dropna(subset=["value"])
        out = out[out["desc"].ne("") & out["desc"].ne("nan")]

        return out.reset_index(drop=True)
    except Exception as e:
        print(f"   ⚠️ Error cleaning data: {e}")
        return pd.DataFrame()

# ==========================================================
# TRANSFORM
# ==========================================================
def transform_to_standard(df_desc_val: pd.DataFrame, title: str, filename: str,
                          year: int, month_name: str) -> pd.DataFrame:
    rows = []
    is_tsc = ("-TSC-" in filename) and ("TSCC" not in filename)
    indicator_type = "Main market" if is_tsc else "Main product"

    for _, r in df_desc_val.iterrows():
        desc = str(r["desc"]).strip()
        val = r["value"]

        dlow = desc.lower()
        if dlow in ["country", "description", "total", "grand total"] or desc.upper().startswith("TOTAL"):
            continue

        rows.append({
            "No.": 0,
            "Title": title,
            "Country": "Cambodia",
            "Update frequency": "Monthly",
            "Status": "Real",
            "Yearly": int(year),
            "Monthly": month_name,
            "Indicator": indicator_type,
            "Sub1": desc if is_tsc else ".",
            "Sub2": desc if (not is_tsc) else ".",
            "Sub3": ".", "Sub4": ".", "Sub5": ".", "Sub6": ".",
            "Unit": "(Value in Thousand US $)",
            "Value": float(val),
            "Access Date": datetime.now().strftime("%m/%d/%Y"),
            "Publish Date": f"{month_name} {year}",
            "Link(if available)": "https://www.customs.gov.kh",
             "Note": " "
        })

    return pd.DataFrame(rows)

# ==========================================================
# MAIN
# ==========================================================
def main():
    if not os.path.exists(EXCEL_PATH):
        print(f"❌ File not found: {EXCEL_PATH}")
        return

    target = subtract_months(first_day_of_month(datetime.now()), MONTHS_BACK)
    year = target.year
    month_num = f"{target.month:02d}"
    month_name = target.strftime("%B")
    date_key = f"{year}-{month_num}"

    print("===================================================")
    print("Excel path:", os.path.abspath(EXCEL_PATH))
    print("Target month:", f"{date_key} ({month_name} {year})")
    print("===================================================")

    headers = {"User-Agent": "Mozilla/5.0", "x-api-token": API_TOKEN}

    # Load workbook ONCE (preserve other sheets)
    try:
        wb = load_workbook(EXCEL_PATH)
    except PermissionError:
        print("❌ Permission denied opening Excel file.")
        print("✅ Close Excel / OneDrive preview and run again.")
        return

    print("Sheets in workbook:", wb.sheetnames)

    for title, desired_sheet_name in SHEET_MAP.items():
        sheet_name = resolve_sheet_name(wb, desired_sheet_name)
        if sheet_name not in wb.sheetnames:
            wb.create_sheet(sheet_name)
        ws = wb[sheet_name]

        print(f"\n--- Processing {title} -> {sheet_name} for {month_name} {year} ---")

        df_existing = sheet_to_df(ws)

        suffix = "EX" if title == "Export" else "IM"
        filenames = [
            f"{date_key}-TSC-EN-{suffix}.xlsx",
            f"{date_key}-TSCC-EN-{suffix}.xlsx",
        ]

        new_dfs = []
        for fname in filenames:
            print(f"🔍 Requesting: {fname}")
            content = download_excel_file(headers, year, fname)
            if not content:
                continue

            raw = clean_raw_data(content)
            print(f"   Parsed rows: {len(raw)}")
            if raw.empty:
                continue

            df_std = transform_to_standard(raw, title, fname, year, month_name)
            print(f"   ✅ Standard rows created: {len(df_std)}")
            if not df_std.empty:
                new_dfs.append(df_std)

        # IMPORTANT: if no new data -> DO NOT remove old data
        if not new_dfs:
            print("ℹ️ No new data created. Keeping existing sheet unchanged.")
            continue

        df_new = pd.concat(new_dfs, ignore_index=True)

        # Remove existing rows for that month ONLY NOW (after new data is ready)
        df_updated = df_existing.copy()
        if not df_updated.empty and "Yearly" in df_updated.columns and "Monthly" in df_updated.columns:
            df_updated["Yearly"] = pd.to_numeric(df_updated["Yearly"], errors="coerce")
            df_updated["Monthly"] = df_updated["Monthly"].astype(str).str.strip()

            mask = (df_updated["Yearly"] == year) & (df_updated["Monthly"] == month_name)
            if mask.any() and FORCE_OVERWRITE:
                print(f"🗑️ Removing {int(mask.sum())} existing rows for {month_name} {year} (overwrite enabled)")
                df_updated = df_updated[~mask].copy()

        # Append and reindex
        df_updated = pd.concat([df_updated, df_new], ignore_index=True) if not df_updated.empty else df_new
        df_updated["No."] = range(1, len(df_updated) + 1)

        # Write back ONLY this sheet
        df_to_sheet(ws, df_updated)
        print(f"✨ Updated {sheet_name}: +{len(df_new)} rows, total={len(df_updated)}")

    # Save (try original, fallback to _updated)
    try:
        wb.save(EXCEL_PATH)
        print("\n🎉 Saved successfully to original file.")
    except PermissionError:
        out_path = make_updated_path(EXCEL_PATH)
        wb.save(out_path)
        print(f"\n⚠️ Original file locked. Saved to: {out_path}")

if __name__ == "__main__":
    main()
